# Delivery bottlenecks, anomalies & statistical tests

This notebook uses the project KPI helpers in `src/metrics.py` to identify dispatch-wait bottlenecks, attribute lateness to depot wait versus downstream delay, flag unusually long waits, and test whether route outcomes differ by stop-sequencing algorithm or dispatch window.

The supplied data is simulated. Tests describe differences in this dataset; they do not establish that an algorithm or dispatch window causes better outcomes in a real fleet. Routes, rather than individual stops, are the unit of analysis for hypothesis tests so routes with more stops do not get extra statistical weight.

In [ ]:
from itertools import combinations
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Resolve the project root whether Jupyter was started from the project or notebooks folder.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / "src" / "metrics.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate src/metrics.py from the notebook working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.metrics import (
    LATE_THRESHOLD_MIN,
    WINDOW_ORDER,
    build_stop_facts,
    delay_attribution,
    flag_outliers_by_group,
    flag_outliers_iqr,
    flag_outliers_zscore,
    driver_clustered_gee,
    load_tables,
    route_kpis,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
routes, stops = load_tables(RAW_DIR)
if routes.empty or stops.empty:
    raise ValueError(f"Expected non-empty route and stop data in {RAW_DIR}")

facts = build_stop_facts(routes, stops, threshold=LATE_THRESHOLD_MIN)
route_metrics = route_kpis(facts)
print(f"Project: {PROJECT_ROOT}")
print(f"Routes: {len(route_metrics):,} | Stops: {len(facts):,} | Late threshold: > {LATE_THRESHOLD_MIN} min")
print(f"Overall late-stop rate: {facts['is_late'].mean():.1%}")

## 1. Locate delay bottlenecks

`late_due_to_wait_rate` is the share of stops that are late but would not exceed the late threshold after subtracting their route's dispatch wait. `late_rate_without_wait` is the share still late after that subtraction. These are descriptive counterfactual metrics from `metrics.py`, not causal estimates.

In [ ]:
depot_attribution = delay_attribution(facts, "depot").sort_values("late_rate", ascending=False)
window_attribution = delay_attribution(facts, "dispatch_window").reindex(WINDOW_ORDER)
display(depot_attribution)
display(window_attribution)

depot_plot = depot_attribution.reset_index().melt(
    id_vars="depot",
    value_vars=["late_rate", "late_rate_without_wait", "late_due_to_wait_rate"],
    var_name="measure",
    value_name="rate",
)
px.bar(
    depot_plot,
    x="depot",
    y="rate",
    color="measure",
    barmode="group",
    title="Late-stop rate and dispatch-wait attribution by depot",
    labels={"depot": "Depot", "rate": "Share of stops", "measure": "Measure"},
).show()

## 2. Flag unusually long dispatch waits

Outliers are assessed on one observation per route. The global flags help surface depots with systematically high waits; the within-depot IQR flag instead highlights routes that are unusual relative to their own depot.

In [ ]:
route_waits = routes[["route_id", "depot", "dispatch_window", "algorithm", "wait_min"]].drop_duplicates("route_id").copy()
if route_waits["wait_min"].isna().any():
    raise ValueError("Route wait times contain missing values; resolve them before outlier analysis.")

route_waits["iqr_outlier_global"] = flag_outliers_iqr(route_waits["wait_min"])
route_waits["zscore_outlier_global"] = flag_outliers_zscore(route_waits["wait_min"])
route_waits["iqr_outlier_within_depot"] = flag_outliers_by_group(
    route_waits, value_col="wait_min", group_col="depot", method="iqr"
)

wait_summary = route_waits.groupby("depot", observed=True).agg(
    routes=("route_id", "size"),
    mean_wait_min=("wait_min", "mean"),
    median_wait_min=("wait_min", "median"),
    p90_wait_min=("wait_min", lambda values: values.quantile(0.90)),
    global_iqr_outliers=("iqr_outlier_global", "sum"),
    global_zscore_outliers=("zscore_outlier_global", "sum"),
    within_depot_iqr_outliers=("iqr_outlier_within_depot", "sum"),
).sort_values("mean_wait_min", ascending=False)
display(wait_summary.round(2))

display(
    route_waits.loc[route_waits["iqr_outlier_global"]]
    .sort_values("wait_min", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

## 3. Do route outcomes differ by algorithm or dispatch window?

The outcome is each route's late-stop rate. The algorithm comparison uses a two-sided Mann–Whitney U test (`cluster_nn` versus `fifo`). Dispatch windows are first compared together with a Kruskal–Wallis test; pairwise Mann–Whitney tests then compare windows, with Holm adjustment within the post-hoc family. Holm-adjusted p-values are reported alongside the raw values.

In [ ]:
algorithm_samples = {
    name: group["late_rate"].dropna().to_numpy()
    for name, group in route_metrics.groupby("algorithm", observed=True)
}
required_algorithms = {"fifo", "cluster_nn"}
if not required_algorithms.issubset(algorithm_samples):
    raise ValueError(f"Expected algorithm groups {sorted(required_algorithms)}; found {sorted(algorithm_samples)}")

fifo = algorithm_samples["fifo"]
cluster_nn = algorithm_samples["cluster_nn"]
algorithm_test = mannwhitneyu(fifo, cluster_nn, alternative="two-sided", method="auto")

window_samples = {
    name: group["late_rate"].dropna().to_numpy()
    for name, group in route_metrics.groupby("dispatch_window", observed=True)
}
ordered_windows = [name for name in WINDOW_ORDER if name in window_samples and len(window_samples[name]) > 0]
if len(ordered_windows) < 2:
    raise ValueError("At least two dispatch windows with route outcomes are required for testing.")
window_test = kruskal(*(window_samples[name] for name in ordered_windows))

primary_tests = pd.DataFrame(
    [
        {
            "comparison": "cluster_nn vs fifo (route late rate)",
            "test": "Mann–Whitney U",
            "statistic": algorithm_test.statistic,
            "p_value": algorithm_test.pvalue,
        },
        {
            "comparison": "all dispatch windows (route late rate)",
            "test": "Kruskal–Wallis",
            "statistic": window_test.statistic,
            "p_value": window_test.pvalue,
        },
    ]
)
primary_tests["p_holm"] = multipletests(primary_tests["p_value"], method="holm")[1]

algorithm_summary = route_metrics.groupby("algorithm", observed=True)["late_rate"].agg(
    routes="size", mean_late_rate="mean", median_late_rate="median"
).round(3)
window_summary = route_metrics.groupby("dispatch_window", observed=True)["late_rate"].agg(
    routes="size", mean_late_rate="mean", median_late_rate="median"
).reindex(ordered_windows).round(3)
display(algorithm_summary)
display(window_summary)
display(primary_tests.style.format({"statistic": "{:.3f}", "p_value": "{:.3e}", "p_holm": "{:.3e}"}))
print(f"Mean route late-rate difference (cluster_nn - fifo): {cluster_nn.mean() - fifo.mean():+.3f}")

In [ ]:
pairwise_rows = []
for first, second in combinations(ordered_windows, 2):
    first_values, second_values = window_samples[first], window_samples[second]
    result = mannwhitneyu(first_values, second_values, alternative="two-sided", method="auto")
    pairwise_rows.append(
        {
            "comparison": f"{first} vs {second}",
            "mean_rate_first": first_values.mean(),
            "mean_rate_second": second_values.mean(),
            "mean_difference_first_minus_second": first_values.mean() - second_values.mean(),
            "statistic": result.statistic,
            "p_value": result.pvalue,
        }
    )

pairwise_tests = pd.DataFrame(pairwise_rows)
if not pairwise_tests.empty:
    pairwise_tests["p_holm"] = multipletests(pairwise_tests["p_value"], method="holm")[1]
    display(
        pairwise_tests.sort_values("p_holm").reset_index(drop=True).style.format(
            {
                "mean_rate_first": "{:.3f}",
                "mean_rate_second": "{:.3f}",
                "mean_difference_first_minus_second": "{:+.3f}",
                "statistic": "{:.1f}",
                "p_value": "{:.3e}",
                "p_holm": "{:.3e}",
            }
        )
    )

## Driver-clustered adjusted comparison

Stops from the same driver are not independent observations. This logistic generalized estimating equation (GEE) clusters observations by driver and adjusts for dispatch window, vehicle type, region, depot, dispatch wait, and route distance. Coefficients are log-odds differences relative to the model's reference categories. The model accounts for within-driver dependence, but it remains observational and cannot rule out unmeasured confounding or establish causality.

In [ ]:
gee_result = driver_clustered_gee(facts)
gee_table = pd.DataFrame(
    {
        "log_odds": gee_result.params,
        "ci_lower": gee_result.conf_int()[0],
        "ci_upper": gee_result.conf_int()[1],
        "p_value": gee_result.pvalues,
    }
)
display(gee_table.style.format({"log_odds": "{:+.3f}", "ci_lower": "{:+.3f}", "ci_upper": "{:+.3f}", "p_value": "{:.3e}"}))

## Interpretation notes

- A higher depot wait and a higher `late_due_to_wait_rate` together point to dispatch as an operational bottleneck; persistent delays after subtracting wait suggest downstream travel, service, or schedule issues.
- IQR and Z-score flags are investigation queues, not proof of bad performance. A global flag and a within-depot flag answer different questions.
- The rank tests compare observed route-level distributions without adjusting for driver clustering or route mix. The GEE accounts for repeated stops from the same driver and adjusts for included route characteristics; omitted variables may still confound its estimates.
- A small p-value indicates evidence of a difference under a model's assumptions, not operational importance or cause. Review the effect estimates, confidence intervals, sample sizes, and percentage-point differences as well.
- For causal decisions, use a randomized or carefully controlled rollout and account for repeated routes per driver and date.